In [2]:
%pip install -U datasets

  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached pyarrow-25.0.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
Using cached pyarrow-25.0.1-cp311-cp311-manylinux_2_28_x86_64.whl (50.1 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 15.0.2
    Uninstalling pyarrow-15.0.2:
      Successfully uninstalled pyarrow-15.0.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.2.1
    Uninstalling datasets-2.2.1:
      Successfully uninstalled datasets-2.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 2.15.1 requires pyarrow<16,>=4.0.0, but you have pyarrow 25.0.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [1]:
# 1. Imports and setup
import sagemaker
import boto3
import pandas as pd
import numpy as np
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

session = sagemaker.Session()
role = get_execution_role()
bucket = session.default_bucket()
prefix = "sagemaker-bitext-support-demo"

# 2. Load the Bitext Customer Support dataset from Hugging Face
#   (run this in a notebook cell first if not already installed)
from datasets import load_dataset

hf_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = hf_dataset["train"].to_pandas()

print(df.columns.tolist())
print(df["category"].value_counts())
df.head()

# 3. Select relevant columns
# 'instruction' = customer message text, 'category' = high-level support category (target)
df = df[["instruction", "category"]].dropna()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
['flags', 'instruction', 'category', 'intent', 'response']
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
CONTACT         1999
INVOICE         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64


In [2]:
# 4. Encode target labels (multiclass)
label_encoder = LabelEncoder()
df["target"] = label_encoder.fit_transform(df["category"])
num_classes = df["target"].nunique()
print(f"Number of classes: {num_classes}")
print(dict(zip(label_encoder.classes_, range(num_classes))))

# 5. Train/validation split (on raw text, before vectorizing)
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["target"]
)

# 6. TF-IDF vectorize the text (fit on train only, transform both)
vectorizer = TfidfVectorizer(max_features=300, stop_words="english")
X_train = vectorizer.fit_transform(train_df["instruction"]).toarray()
X_val = vectorizer.transform(val_df["instruction"]).toarray()

# 7. Build final CSV: target column first, then TF-IDF features
train_final = pd.DataFrame(X_train)
train_final.insert(0, "target", train_df["target"].values)

val_final = pd.DataFrame(X_val)
val_final.insert(0, "target", val_df["target"].values)

train_final.to_csv("train.csv", header=False, index=False)
val_final.to_csv("validation.csv", header=False, index=False)

# 8. Upload to S3
train_path = session.upload_data("train.csv", bucket=bucket, key_prefix=f"{prefix}/train")
val_path = session.upload_data("validation.csv", bucket=bucket, key_prefix=f"{prefix}/validation")

train_input = TrainingInput(train_path, content_type="text/csv")
val_input = TrainingInput(val_path, content_type="text/csv")

# 9. Get the built-in XGBoost container image
from sagemaker.image_uris import retrieve
xgboost_image = retrieve("xgboost", session.boto_region_name, version="1.7-1")

Number of classes: 11
{'ACCOUNT': 0, 'CANCEL': 1, 'CONTACT': 2, 'DELIVERY': 3, 'FEEDBACK': 4, 'INVOICE': 5, 'ORDER': 6, 'PAYMENT': 7, 'REFUND': 8, 'SHIPPING': 9, 'SUBSCRIPTION': 10}


In [3]:
# 10. Configure the estimator for MULTICLASS classification
xgb_estimator = sagemaker.estimator.Estimator(
    image_uri=xgboost_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",   # training instance — t3 not supported for training
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session,
)

xgb_estimator.set_hyperparameters(
    objective="multi:softprob",
    num_class=num_classes,
    num_round=150,
    max_depth=5,
    eta=0.2,
    eval_metric="mlogloss",
)

In [4]:
# 11. Train the model
xgb_estimator.fit({"train": train_input, "validation": val_input})
# 12. Deploy to a lightweight endpoint (t3.medium works fine for hosting)
predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
)

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-08-23-11-52-56-131


2026-08-23 11:52:57 Starting - Starting the training job...
2026-08-23 11:53:10 Starting - Preparing the instances for training...
2026-08-23 11:53:32 Downloading - Downloading input data...
2026-08-23 11:54:22 Downloading - Downloading the training image......
2026-08-23 11:55:28 Training - Training image download completed. Training in progress.../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-08-23 11:55:32.013 ip-10-0-187-93.ap-south-1.compute.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-08-23 11:55:32.124 ip-10-0-187-93.ap-south-1.compute.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-08-23:11:55:32:INFO] Imported framework sage

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-08-23-12-01-48-523


Training seconds: 486
Billable seconds: 486


INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2026-08-23-12-01-48-523
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2026-08-23-12-01-48-523


------!

In [8]:
# 13. Run inference on a few validation samples
from sagemaker.serializers import CSVSerializer
predictor.serializer = CSVSerializer()

sample = val_final.drop(columns=["target"]).iloc[:5].values
result = predictor.predict(sample)
print("Predicted class probabilities:\n", result)

predicted_labels = val_final["target"].iloc[:5].values
actual_categories = label_encoder.inverse_transform(predicted_labels)
print("Actual categories:", actual_categories)

Predicted class probabilities:
 b'0.9940471649169922,0.0003524710482452065,0.0009615299059078097,0.0003929717931896448,0.00046981146442703903,0.0010212109191343188,0.0010878854664042592,0.00028895289869979024,0.000816882704384625,0.0003952879342250526,0.00016587632126174867\n5.8375560911372304e-05,3.2990199088089867e-06,2.1476684196386486e-05,1.429848180123372e-05,2.7505378966452554e-05,4.146130322624231e-06,4.6856188419042155e-05,1.0583577022771351e-05,2.002330256800633e-05,0.9997579455375671,3.544721403159201e-05\n1.0278354238835163e-05,6.843429218861274e-07,2.495078433639719e-06,1.150803313976212e-06,1.3758253771811724e-06,9.18449359232909e-07,3.185831928931293e-06,8.461878451271332e-07,0.9999774694442749,1.1021350019291276e-06,4.857625981458114e-07\n8.14650920801796e-05,1.1083821846114006e-05,0.9996116757392883,2.525199670344591e-05,0.00011106862802989781,7.806508619978558e-06,4.3732226913562045e-05,2.0443860194063745e-05,6.731489702360705e-05,1.3088986634102184e-05,7.0507235250261

In [9]:
# 14. Clean up (avoid ongoing charges)
predictor.delete_endpoint()

INFO:sagemaker:Deleting endpoint configuration with name: sagemaker-xgboost-2026-08-23-12-01-48-523
INFO:sagemaker:Deleting endpoint with name: sagemaker-xgboost-2026-08-23-12-01-48-523
